In [1]:
import os
import json
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
from collections import defaultdict

In [2]:
STAGE1_ROOT = Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/intermediate")
OUTPUT_ROOT = Path("/home/abhishek/Desktop/projects/RAG_MOLE/data/processed")


In [3]:
g1_files = sorted((STAGE1_ROOT/"G1").rglob("*.json"))
g2_files = sorted((STAGE1_ROOT/"G2").rglob("*.json"))
g3_files = sorted((STAGE1_ROOT/"G3").rglob("*.json"))
all_files = []
all_files.extend(g1_files)
all_files.extend(g2_files)
all_files.extend(g3_files)
print(f"Total files : {len(all_files)}")

Total files : 83761


In [33]:
sample_file = all_files[0]

with open(sample_file, "r", encoding="utf-8") as f:
    sample = json.load(f)

print(json.dumps(sample, indent=4))

sample2_file=g3_files[0]
with open(sample2_file, "r", encoding="utf-8") as f:
    sample2 = json.load(f)

print(json.dumps(sample2, indent=4))
print(sample2_file)

{
    "query": "I am a food blogger and I would like to create a recipe using a specific ingredient. Can you provide me with a list of recipes that include this ingredient, along with their preparation instructions and cooking time? It would be great if you could also fetch photos of the completed dishes.",
    "trace": [
        {
            "step": 1,
            "api_name": "url_for_newapi",
            "arguments_raw": "{\n  \"url\": \"https://www.example.com/recipes?ingredient=specific_ingredient\"\n}",
            "response_raw": "{\"error\": \"\", \"response\": \"{'message': \\\"Endpoint '/https:/www.example.com/recipes' does not exist\\\"}\"}",
            "observation_code": 0
        },
        {
            "step": 2,
            "api_name": "n_for_newapi",
            "arguments_raw": "{\n  \"language\": \"english\"\n}",
            "response_raw": "{\"error\": \"Message error...\", \"response\": \"<!doctype html>\\n<html lang=en>\\n<title>500 Internal Server Error</title>

In [29]:
def load_stage1_sample(file_path):
    

    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

In [34]:
def extract_semantic_context(stage1_json):
    """
    Converts Stage1 JSON into semantic evidence for the Intent Compiler.
    No API mapping is performed.
    """

    context = {}

    context["query"] = stage1_json.get("query", "")
    context["goal"] = stage1_json.get("goal", "")
    context["final_answer"] = stage1_json.get("final_answer", "")

    context["trace"] = []

    for node in stage1_json["trace"]:

        try:
            arguments = json.loads(node["arguments_raw"])
        except Exception:
            arguments = node["arguments_raw"]

        context["trace"].append({

            "step": node["step"],

            "api_name": node["api_name"],

            "arguments": arguments,

            "response": node["response_raw"]

        })

    return context

In [35]:
context = extract_semantic_context(sample)

print(json.dumps(context, indent=4))
context2 = extract_semantic_context(sample2)
print(json.dumps(context2, indent=4))

{
    "query": "I am a food blogger and I would like to create a recipe using a specific ingredient. Can you provide me with a list of recipes that include this ingredient, along with their preparation instructions and cooking time? It would be great if you could also fetch photos of the completed dishes.",
    "goal": "",
    "final_answer": "I'm sorry, but I'm currently unable to fetch the recipes. Please try again later.",
    "trace": [
        {
            "step": 1,
            "api_name": "url_for_newapi",
            "arguments": {
                "url": "https://www.example.com/recipes?ingredient=specific_ingredient"
            },
            "response": "{\"error\": \"\", \"response\": \"{'message': \\\"Endpoint '/https:/www.example.com/recipes' does not exist\\\"}\"}"
        },
        {
            "step": 2,
            "api_name": "n_for_newapi",
            "arguments": {
                "language": "english"
            },
            "response": "{\"error\": \"Messa

In [36]:
stats = defaultdict(int)

for file in tqdm(all_files):

    sample = load_stage1_sample(file)

    stats["files"] += 1

    stats["capabilities"] += len(sample["trace"])

print("="*40)

for k,v in stats.items():

    print(f"{k:<20} : {v}")

  0%|          | 0/83761 [00:00<?, ?it/s]

files                : 83761
capabilities         : 249619


In [ ]:
import random

for _ in range(3):

    sample = load_stage1_sample(random.choice(all_files))

    print("="*80)

    print(sample["query"])

    print()

    for t in sample["trace"]:

        print(t["api_name"])

I'm a teacher planning a lesson on climate change and I need a variety of articles from different sources. Can you provide me with articles from publications like Nasa Climate, United Nations, and Carbon Brief? Please exclude articles from The Guardian and limit the articles to 50. It would be helpful if you could provide me with the publication name, article title, and the date it was published.

get_articles_for_climate_news_feed
I'm hosting a company event and need to find a suitable venue. Can you suggest some event venues in the city that can accommodate a large number of people? Additionally, find some elegant anime images that I can use for the event invitations.

getinventory_for_tes
anime_for_any_anime
My company is planning to renew our domains. Before proceeding, I need to check the domain details for '576517' and get a complete list of our domains, including their descriptions, extensions, years, discounts, start dates, paid until dates, external statuses, redirect URLs, DN

In [37]:
def build_compiler_input(context):

    return {

        "query": context["query"],

        "goal": context["goal"],

        "trace": context["trace"],

        "final_answer": context["final_answer"]

    }

In [38]:
compiler_input = build_compiler_input(context)

print(json.dumps(compiler_input, indent=4))

{
    "query": "I am a food blogger and I would like to create a recipe using a specific ingredient. Can you provide me with a list of recipes that include this ingredient, along with their preparation instructions and cooking time? It would be great if you could also fetch photos of the completed dishes.",
    "goal": "",
    "trace": [
        {
            "step": 1,
            "api_name": "url_for_newapi",
            "arguments": {
                "url": "https://www.example.com/recipes?ingredient=specific_ingredient"
            },
            "response": "{\"error\": \"\", \"response\": \"{'message': \\\"Endpoint '/https:/www.example.com/recipes' does not exist\\\"}\"}"
        },
        {
            "step": 2,
            "api_name": "n_for_newapi",
            "arguments": {
                "language": "english"
            },
            "response": "{\"error\": \"Message error...\", \"response\": \"<!doctype html>\\n<html lang=en>\\n<title>500 Internal Server Error</title

In [39]:
SYSTEM_PROMPT = """
You are an expert Semantic Intent Compiler.

Your task is to infer the canonical provider-independent semantic intents
required to satisfy a user's request.

You are NOT an execution planner.

You are NOT an API selector.

You are NOT a registry lookup system.

Your job is ONLY to understand the user's semantic requirements.

INPUT

You will receive:

1. User Query
2. Goal
3. Execution Trace
4. Final Answer

The execution trace and final answer are only supporting evidence.

The user query is the primary source of truth.

Rules

- Infer only semantic intents explicitly grounded in the user's request.
- Use the execution trace only to understand how the request was interpreted.
- Do NOT copy API names.
- Do NOT copy providers.
- Do NOT copy registry ids.
- Do NOT copy endpoints.
- Do NOT infer implementation strategies.
- Do NOT decide how many APIs are required.
- Do NOT merge unrelated intents.
- Normalize synonymous intents into a canonical form.
- Return ONLY JSON.

Output Schema

{
    "goal": "...",
    "intents": [
        "...",
        "...",
        "..."
    ]
}
"""

In [46]:
def build_user_prompt(context):

    prompt = f"""
USER QUERY
==========

{context["query"]}


EXECUTION TRACE
===============

{json.dumps(context["trace"], indent=4)}


FINAL ANSWER
============

{context["final_answer"]}


TASK
====

Infer the canonical semantic intents required to satisfy the user's request.

The user query is the primary source of truth.

The execution trace and final answer are supporting evidence only.

Ignore:

- API names
- Tool names
- Providers
- Registry IDs
- Parameters
- Execution strategy

Return ONLY valid JSON.

"""

    return prompt

In [47]:
compiler_prompt = build_user_prompt(context)
compiler_prompt2 = build_user_prompt(context2)
print(compiler_prompt)
print (compiler_prompt2)


USER QUERY

I am a food blogger and I would like to create a recipe using a specific ingredient. Can you provide me with a list of recipes that include this ingredient, along with their preparation instructions and cooking time? It would be great if you could also fetch photos of the completed dishes.


EXECUTION TRACE

[
    {
        "step": 1,
        "api_name": "url_for_newapi",
        "arguments": {
            "url": "https://www.example.com/recipes?ingredient=specific_ingredient"
        },
        "response": "{\"error\": \"\", \"response\": \"{'message': \\\"Endpoint '/https:/www.example.com/recipes' does not exist\\\"}\"}"
    },
    {
        "step": 2,
        "api_name": "n_for_newapi",
        "arguments": {
            "language": "english"
        },
        "response": "{\"error\": \"Message error...\", \"response\": \"<!doctype html>\\n<html lang=en>\\n<title>500 Internal Server Error</title>\\n<h1>Internal Server Error</h1>\\n<p>The server encountered an internal 

In [42]:
from openai import OpenAI

client = OpenAI(

    base_url="http://localhost:11434/v1",

    api_key="ollama"

)

In [43]:
def compile_intent(context, model="qwen3:8b"):

    response = client.chat.completions.create(

        model=model,

        messages=[

            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": build_user_prompt(context)
            }

        ],

        temperature=0,
        extra_body={
            "think": False
        }

    )

    return response.choices[0].message.content

In [45]:

result = compile_intent(context2)

print(result)

{
    "goal": "Provide information about Los Angeles for a family vacation planning",
    "intents": [
        "Retrieve city demographics (population, area)",
        "Obtain timezone information",
        "Get current weather forecast",
        "Identify top tourist attractions"
    ]
}


In [18]:
def parse_compiler_output(output):

    try:

        return json.loads(output)

    except json.JSONDecodeError as e:

        print(e)

        return None

In [19]:
def parse_compiler_output(output):

    try:

        return json.loads(output)

    except json.JSONDecodeError as e:

        print(e)

        return None

In [ ]:
def validate_discovery_schema(sample):

    if not isinstance(sample, dict):
        return False

    if "goal" not in sample:
        return False

    if "intents" not in sample:
        return False

    if not isinstance(sample["intents"], list):
        return False

    required_fields = {

    "goal",

    "sub_intents",

    "intent_graph"

    }

    for intent in sample["intents"]:

        if not required_fields.issubset(intent.keys()):

            return False

    return True

In [21]:
print(validate_discovery_schema(parsed))

NameError: name 'parsed' is not defined

In [36]:
from ollama import chat
import time

start = time.time()

response = chat(
    model="qwen3:8b",
    messages=[
        {"role": "user", "content": "Hi"}
    ]
)

print(response["message"]["content"])
print("Time:", time.time() - start)

Hello! How can I assist you today? 😊
Time: 2.3981587886810303


In [ ]:
from openai import OpenAI
import time

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)Currently your compiler is constrained by



start = time.time()

response = client.chat.completions.create(
    model="qwen3:8b",
    messages=[
        {"role": "user", "content": "Hi"}
    ],
    temperature=0
)

print(response.choices[0].message.content)
print("Time:", time.time() - start)


Hello! 😊 How can I assist you today? I'm here to help with any questions or tasks you might have!
Time: 2.8910224437713623
